# OpenJev: fine-tune trên Colab

Prompt v2: định nghĩa và ranh giới nằm ngay trong từng câu hỏi identity/scope. Giữ nguyên cách đọc A/B và nhãn gold. Dùng thư mục kết quả mới để so sánh với prompt v1; không tái sử dụng ngưỡng đã chọn từ xác suất v1.

Chọn **Runtime → Change runtime type → GPU** (A100 khuyến nghị). Upload các TSV vào session, rồi sửa đường dẫn ở phần 2. Không mount Drive; không cần GPU local. Notebook clone nhánh GitHub vào `/content/semeval`. Trên Kaggle, code tự dùng `/kaggle/working/semeval`; bật Internet và GPU rồi sửa đường dẫn input.

Train **LoRA BF16**, không quantize. Input chỉ gồm title + description + comment. Mỗi identity là một câu yes/no; scope là một câu chọn individual/group. Giữ nguyên split gốc; dữ liệu bổ sung chỉ ghép trong bộ nhớ. Dev tổng hợp được báo cáo riêng. Phiên bản này chọn ngưỡng chung, chưa hiệu chỉnh calibration.

## 1. Lấy code và cài thư viện
Chạy cell cài đặt trước khi import torch/transformers. Nếu đã import các thư viện đó trong session trước khi cài, khởi động lại session rồi chạy notebook từ đầu.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = "https://github.com/iamtro-2006/semeval.git"
BRANCH = "main"
BASE = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
REPO_DIR = BASE / "semeval"
BASE.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO, str(REPO_DIR)], check=True)
else:
    current = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
    changes = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
    if current != BRANCH or changes:
        raise RuntimeError("Existing clone has a different branch or local edits. Choose a new REPO_DIR.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

WORK = REPO_DIR / "task 8" / "openjev"
assert (WORK / "run.py").is_file(), WORK
print("Code:", WORK)
subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], check=True)


In [ ]:
import importlib.metadata
from packaging.version import Version

packages = ["transformers==5.17.0", "peft==0.19.1", "accelerate==1.13.0",
            "safetensors>=0.7,<1", "tqdm>=4.67,<5"]
try:
    torch_version = Version(importlib.metadata.version("torch").split("+")[0])
except importlib.metadata.PackageNotFoundError:
    torch_version = Version("0")
# Preserve a recent Colab/Kaggle PyTorch stack. Upgrade the matched trio only if needed.
if torch_version < Version("2.8"):
    packages += ["torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

import importlib.util
spec = importlib.util.spec_from_file_location("openjev_run", WORK / "run.py")
run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run)
print("Ready. Model weights have not been downloaded yet.")


## 2. Đường dẫn và cấu hình
Upload `train.tsv`, `dev.tsv`, tùy chọn `test.tsv` vào session. TSV cần các cột `StereoQueerEval_id`, `yt_title`, `yt_description`, `yt_comment`, `target`; các cột khác được bỏ qua. Trên Kaggle, sửa đường dẫn thành `/kaggle/input/...`.

Để `TEST = ""` trong lúc phát triển; chỉ thêm test sau khi chốt thiết kế. Không dùng output folder của lần chạy khác. `RESUME=True` yêu cầu đã có hoặc khôi phục `checkpoint/state.pt` của chính lần chạy này; cấu hình và dữ liệu phải giữ nguyên.


In [ ]:
TRAIN = str(BASE / "train.tsv")
DEV = str(BASE / "dev.tsv")
TEST = ""  # e.g. /content/test.tsv, or /kaggle/input/your-dataset/test.tsv
OUTPUT = str(BASE / "results-v2")

MODEL = "openjev/OpenJev-Flash-9B"
EPOCHS = 2
MAX_LENGTH = 2048
LORA_RANK = 8
LEARNING_RATE = 1e-4
ACCUMULATION = 16  # GPU micro-batch is 1; this sets effective batch size
SAVE_STEPS = 100
ADD_SAMPLES = True  # 4 a-positive + 2 negative controls per train/dev split
RESUME = False

config = dict(train=TRAIN, dev=DEV, test=TEST, output=OUTPUT, model=MODEL,
              epochs=EPOCHS, max_length=MAX_LENGTH, rank=LORA_RANK, lr=LEARNING_RATE,
              accumulation=ACCUMULATION, save_steps=SAVE_STEPS, samples=ADD_SAMPLES)
job = run.Run(config)


## 3. Kiểm tra trước khi train
Kiểm tra GPU, độ dài **tất cả** prompt và tokenizer. Không tự cắt comment/description. Nếu có prompt quá dài, cell báo độ dài cần thiết để bạn chỉnh `MAX_LENGTH` rồi chạy lại từ phần 2. Các câu hỏi của cùng comment luôn giữ nguyên split.

Train/dev có video trùng nhau sẽ được thống kê theo protocol split hiện có. Mẫu tổng hợp có đánh dấu nguồn; không được xem là gold chính thức. Tập gốc hiện chỉ có hateful target nên chưa đủ để đánh giá hate/non-hate trên dữ liệu thật.


In [ ]:
job.check()


## 4. Fine-tune LoRA BF16
Cell này tải checkpoint BF16, đóng băng weights nền và phần ảnh, rồi gắn LoRA vào các module văn bản. Trước khi train, tự chạy forward/backward trên prompt train dài nhất và in peak VRAM; lỗi GPU/memory được giữ nguyên để xử lý.

Checkpoint tốt nhất được chọn bằng loss trên **dev gốc**. Chỉ giữ một snapshot training gần nhất (mỗi 100 optimizer steps theo cấu hình), chứa cả optimizer và RNG để tiếp tục. Colab ngắt trước lần lưu kế tiếp sẽ mất phần tiến độ chưa lưu. `adapter/` lưu adapter của epoch tốt nhất, không lưu toàn bộ weights 9B.


In [ ]:
job.train(resume=RESUME)


## 5. Chọn ngưỡng và đánh giá
Một ngưỡng chung được chọn theo identity Micro-F1 trên **dev gốc**, sau đó cố định trước khi dự đoán test. Điểm dev là điểm đã dùng để chọn model/ngưỡng; test là đánh giá held-out. Dev tổng hợp chỉ là chẩn đoán riêng, không dùng chọn model/ngưỡng. Bốn mẫu tổng hợp không đủ để khẳng định khả năng nhận diện a trên dữ liệu thật.

Xác suất được đọc từ logits A/B, không sinh JSON hoặc reasoning. Đây chưa phải xác suất đã calibration. Nếu không identity nào vượt ngưỡng, ghi `abstain` và vẫn tính là target không hợp lệ trong metrics; không tự biến thành none hoặc bỏ mẫu.


In [ ]:
metrics = job.evaluate()
print("Results:", OUTPUT)


## 6. Tải kết quả
Chỉ tạo `adapter/`, `checkpoint/`, `settings.json`, `history.csv`, `predictions.csv`, `metrics.json` trong OUTPUT. Không tạo bản sao dataset hoặc weights nền.

Kết quả trong session có thể mất khi session kết thúc. Giữ checkpoint trong ZIP để tiếp tục training ở phiên sau; upload ZIP và giải nén về đúng OUTPUT trước khi bật RESUME. Nếu chuyển môi trường, cần cùng phiên bản thư viện/model, dữ liệu và code. File dự đoán chỉ chứa ID, nhãn và xác suất, không chứa nội dung comment.


In [ ]:
import zipfile

INCLUDE_CHECKPOINT = True
folder = Path(OUTPUT).resolve()
archive = folder.parent / (folder.name + ".zip")
if not (folder / "settings.json").is_file():
    raise FileNotFoundError("No training run to export yet.")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(folder.rglob("*")):
        if not path.is_file() or path.suffix == ".tmp":
            continue
        relative = path.relative_to(folder)
        if not INCLUDE_CHECKPOINT and relative.parts[0] == "checkpoint":
            continue
        z.write(path, arcname=str(relative))
print("ZIP:", archive)

try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
else:
    files.download(str(archive))
